# Local pipeline: Feast + MLflow + CodeCarbon
Predict a CrossFit athlete's deadlift with a random forest.
We build 2 versions of the features and train with 2 sets of hyperparameters, so 4 experiments in total.

In [ ]:
%matplotlib inline
import os
from datetime import datetime

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from codecarbon import OfflineEmissionsTracker
from feast import Entity, FeatureService, FeatureStore, FeatureView, Field, FileSource
from feast.types import Float64
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

os.makedirs('feature_repo/data', exist_ok=True)
os.makedirs('results', exist_ok=True)

## 1. Load and clean the data

In [ ]:
df = pd.read_csv('data/athletes.csv')

# keep only the columns we need and drop rows with missing values
cols = ['athlete_id', 'gender', 'age', 'height', 'weight', 'candj', 'snatch', 'backsq', 'deadlift']
df = df[cols].dropna().drop_duplicates('athlete_id')
df = df[df['gender'].isin(['Male', 'Female'])]

# remove unrealistic values
df = df[df['age'].between(14, 80) & df['height'].between(48, 84) & df['weight'].between(90, 400)]
df = df[df['candj'].between(40, 600) & df['snatch'].between(30, 500)]
df = df[df['backsq'].between(50, 800) & df['deadlift'].between(80, 900)]
df['athlete_id'] = df['athlete_id'].astype(int)

print(len(df), 'athletes after cleaning')

## 2. Create two versions of the features
- **v1:** body measurements only
- **v2:** v1 plus the other lifts and each lift divided by bodyweight

In [ ]:
v1 = df[['athlete_id', 'age', 'height', 'weight']].copy()
v1['is_male'] = (df['gender'] == 'Male').astype(float)
v1['bmi'] = 703 * df['weight'] / df['height'] ** 2

v2 = v1.copy()
for lift in ['candj', 'snatch', 'backsq']:
    v2[lift] = df[lift]
    v2[lift + '_ratio'] = df[lift] / df['weight']

# Feast needs a timestamp column
v1['event_timestamp'] = pd.Timestamp('2020-01-01', tz='UTC')
v2['event_timestamp'] = pd.Timestamp('2020-01-01', tz='UTC')
v1.to_parquet('feature_repo/data/features_v1.parquet', index=False)
v2.to_parquet('feature_repo/data/features_v2.parquet', index=False)

# the label (deadlift) is kept separate and joined in when we train
labels = df[['athlete_id', 'deadlift']].copy()
labels['event_timestamp'] = pd.Timestamp('2021-01-01', tz='UTC')

v2.head()

## 3. Set up the feature store (Feast)

In [ ]:
# Feast config file
with open('feature_repo/feature_store.yaml', 'w') as f:
    f.write('project: athletes\n')
    f.write('registry: data/registry.db\n')
    f.write('provider: local\n')
    f.write('online_store:\n')
    f.write('  type: sqlite\n')
    f.write('  path: data/online.db\n')
    f.write('entity_key_serialization_version: 2\n')

athlete = Entity(name='athlete_id', join_keys=['athlete_id'])

v1_cols = [c for c in v1.columns if c not in ('athlete_id', 'event_timestamp')]
v2_cols = [c for c in v2.columns if c not in ('athlete_id', 'event_timestamp')]

source_v1 = FileSource(path=os.path.abspath('feature_repo/data/features_v1.parquet'),
                       timestamp_field='event_timestamp')
source_v2 = FileSource(path=os.path.abspath('feature_repo/data/features_v2.parquet'),
                       timestamp_field='event_timestamp')

view_v1 = FeatureView(name='athlete_features_v1', entities=[athlete],
                      schema=[Field(name=c, dtype=Float64) for c in v1_cols], source=source_v1)
view_v2 = FeatureView(name='athlete_features_v2', entities=[athlete],
                      schema=[Field(name=c, dtype=Float64) for c in v2_cols], source=source_v2)

service_v1 = FeatureService(name='athlete_service_v1', features=[view_v1])
service_v2 = FeatureService(name='athlete_service_v2', features=[view_v2])

store = FeatureStore(repo_path='feature_repo')
store.apply([athlete, view_v1, view_v2, service_v1, service_v2])
store.materialize(start_date=datetime(2019, 1, 1), end_date=datetime(2022, 1, 1))

## 4. Run the 4 experiments
Same algorithm (random forest) every time. Only the feature version and the hyperparameters change.

In [ ]:
hyperparams = {
    'HP-A': {'n_estimators': 50, 'max_depth': 6},
    'HP-B': {'n_estimators': 100, 'max_depth': 10},
}

mlflow.set_experiment('athletes-deadlift')

for version in ['v1', 'v2']:
    # get the features for this version from the feature store
    service = store.get_feature_service('athlete_service_' + version)
    data = store.get_historical_features(entity_df=labels, features=service).to_df()

    X = data.drop(columns=['athlete_id', 'event_timestamp', 'deadlift'])
    y = data['deadlift']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    for hp_name, params in hyperparams.items():
        run_name = version + '_' + hp_name

        with mlflow.start_run(run_name=run_name):
            # measure the carbon emissions of the training
            tracker = OfflineEmissionsTracker(country_iso_code='USA', save_to_file=False, log_level='error')
            tracker.start()
            model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
            model.fit(X_train, y_train)
            emissions = tracker.stop()

            pred = model.predict(X_test)
            rmse = mean_squared_error(y_test, pred) ** 0.5
            mae = mean_absolute_error(y_test, pred)
            r2 = r2_score(y_test, pred)

            # three plots in one picture
            fig, ax = plt.subplots(1, 3, figsize=(15, 4))
            ax[0].scatter(y_test, pred, s=3, alpha=0.3)
            ax[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
            ax[0].set_title('Predicted vs actual')
            ax[1].hist(y_test - pred, bins=50)
            ax[1].set_title('Residuals')
            bars = ax[2].barh(X.columns, model.feature_importances_)
            ax[2].bar_label(bars, fmt='%.2f', fontsize=8)
            ax[2].margins(x=0.15)
            ax[2].set_title('Feature importance')
            fig.suptitle(run_name)
            plt.tight_layout()
            plt.show()

            mlflow.log_param('feature_version', version)
            mlflow.log_params(params)
            mlflow.log_metrics({'rmse': rmse, 'mae': mae, 'r2': r2, 'emissions_kg': emissions})

            print(run_name, 'rmse =', round(rmse, 2), 'r2 =', round(r2, 3), 'emissions_kg =', emissions)

## 5. Compare the results (metrics and carbon)

In [ ]:
runs = mlflow.search_runs(experiment_names=['athletes-deadlift']).sort_values('start_time')
results = runs[['tags.mlflow.runName', 'metrics.rmse', 'metrics.mae', 'metrics.r2', 'metrics.emissions_kg']]
results.columns = ['run', 'rmse', 'mae', 'r2', 'emissions_kg']
results = results.drop_duplicates('run', keep='last').sort_values('run')   # keep the latest run of each
results.to_csv('results/comparison.csv', index=False)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
bars = ax[0].bar(results['run'], results['rmse'])
ax[0].bar_label(bars, fmt='%.1f')
ax[0].margins(y=0.15)
ax[0].set_title('RMSE (lower is better)')
bars = ax[1].bar(results['run'], results['emissions_kg'] * 1000)
ax[1].bar_label(bars, fmt='%.4f')
ax[1].margins(y=0.15)
ax[1].set_title('Carbon emissions (grams CO2)')
plt.show()

results

## 7. Conclusions
- The feature version matters most. v2 (with the other lifts) cuts the error a lot compared with v1.
- The bigger forest (HP-B) gives almost no improvement on v2 but uses several times more carbon.
- The carbon numbers are estimates, so they are best used to compare the runs with each other.